# Loan Default Risk — Modelling and Evaluation

**Project 4 | Gwachat Kozah**

This notebook covers the full modelling pipeline for the SuperLender loan default prediction task. It builds on the findings from `01_eda.ipynb` and works through:

1. Data loading, merging, and feature engineering
2. Train/validation split
3. Baseline model — Logistic Regression
4. Handling class imbalance — class weights and SMOTE
5. Model comparison — Logistic Regression, Random Forest, XGBoost
6. Hyperparameter tuning — RandomizedSearchCV
7. Threshold optimisation
8. Final model evaluation
9. SHAP explainability
10. Generating the Zindi submission file

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from src.loader import load_and_merge
from src.features import engineer_model_features
from src.model import get_models, get_param_grids, tune_model, save_model, predict_proba
from src.evaluate import evaluate_model, find_optimal_threshold, plot_confusion_matrix, plot_roc_curves, plot_shap_summary

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted")

%matplotlib inline

## 1. Load, Merge, and Engineer Features

In [ ]:
# Load and merge all three training tables
train_df = load_and_merge(split="train")
print(f"Merged training data shape: {train_df.shape}")
train_df.head()

In [ ]:
# Engineer features and extract target
X, y = engineer_model_features(train_df, is_train=True)

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nClass balance: {y.mean()*100:.1f}% Good, {(1-y.mean())*100:.1f}% Bad")

## 2. Train / Validation Split

In [ ]:
# Stratified split to preserve class balance in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:   {X_train.shape} | Bad rate: {(1-y_train.mean())*100:.1f}%")
print(f"Validation set: {X_val.shape}   | Bad rate: {(1-y_val.mean())*100:.1f}%")

## 3. Baseline Model — Logistic Regression

In [ ]:
# Train baseline with default settings and class_weight='balanced'
from sklearn.linear_model import LogisticRegression
from src.model import build_pipeline

baseline = build_pipeline(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    scale=True
)
baseline.fit(X_train, y_train)

baseline_proba = predict_proba(baseline, X_val)
baseline_metrics = evaluate_model(y_val, baseline_proba)

print("Baseline Logistic Regression:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

## 4. Handling Class Imbalance — SMOTE

In [ ]:
# SMOTE generates synthetic minority class samples to balance the training set
# We apply it only to the training data, never to validation or test data
from sklearn.impute import SimpleImputer

# Impute before SMOTE since SMOTE cannot handle NaN
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imputed, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_smote).value_counts().to_dict()}")

## 5. Model Comparison

In [ ]:
# Train all three models with default hyperparameters first
# to establish a pre-tuning baseline for each
models = get_models()
results_pretune = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    proba = predict_proba(pipeline, X_val)
    metrics = evaluate_model(y_val, proba)
    results_pretune[name] = metrics
    print(f"{name}: ROC-AUC={metrics['roc_auc']:.4f} | F1-macro={metrics['f1_macro']:.4f}")

In [ ]:
# Summary table
summary_pretune = pd.DataFrame(results_pretune).T
summary_pretune[["roc_auc", "f1_macro", "f1_bad", "recall_bad", "precision_bad"]]

## 6. Hyperparameter Tuning — RandomizedSearchCV

In [ ]:
# Tune all three models using RandomizedSearchCV with 5-fold stratified CV
# This cell will take several minutes to run
param_grids = get_param_grids()
tuned_models = {}

for name, pipeline in get_models().items():
    search = tune_model(
        name=name,
        pipeline=pipeline,
        param_grid=param_grids[name],
        X_train=X_train,
        y_train=y_train,
        n_iter=30,
        cv=5
    )
    tuned_models[name] = search.best_estimator_

In [ ]:
# Evaluate tuned models on validation set
results_tuned = {}
roc_data = {}

for name, model in tuned_models.items():
    proba = predict_proba(model, X_val)
    metrics = evaluate_model(y_val, proba)
    results_tuned[name] = metrics
    roc_data[name] = {"y_true": y_val, "y_pred_proba": proba}
    print(f"{name}: ROC-AUC={metrics['roc_auc']:.4f} | F1-macro={metrics['f1_macro']:.4f} | Recall-bad={metrics['recall_bad']:.4f}")

In [ ]:
# Summary table — tuned models
summary_tuned = pd.DataFrame(results_tuned).T
summary_tuned[["roc_auc", "f1_macro", "f1_bad", "recall_bad", "precision_bad"]]

In [ ]:
# ROC curves for all tuned models
plot_roc_curves(roc_data)

## 7. Threshold Optimisation

In [ ]:
# Select the best model based on ROC-AUC from tuning results
best_model_name = max(results_tuned, key=lambda k: results_tuned[k]["roc_auc"])
best_model = tuned_models[best_model_name]
best_proba = predict_proba(best_model, X_val)

print(f"Best model: {best_model_name}")

# Find the threshold that maximises F1 for the Bad class
optimal_threshold = find_optimal_threshold(y_val, best_proba)

In [ ]:
# Compare default threshold (0.5) vs optimal threshold
metrics_default = evaluate_model(y_val, best_proba, threshold=0.5)
metrics_optimal = evaluate_model(y_val, best_proba, threshold=optimal_threshold)

comparison = pd.DataFrame({
    "threshold=0.5": metrics_default,
    f"threshold={optimal_threshold:.3f}": metrics_optimal
}).T
comparison[["roc_auc", "f1_macro", "f1_bad", "recall_bad", "precision_bad"]]

In [ ]:
# Confusion matrix at optimal threshold
y_pred_optimal = (best_proba >= optimal_threshold).astype(int)
plot_confusion_matrix(y_val, y_pred_optimal, model_name=best_model_name)

## 8. Save Best Model

In [ ]:
save_model(best_model, name=best_model_name)

## 9. SHAP Explainability

In [ ]:
# SHAP explains which features push a prediction towards Good or Bad
# We use the validation set for explanation
plot_shap_summary(best_model, X_val, model_name=best_model_name)

## 10. Generate Zindi Submission

In [ ]:
# Load and process test data
test_df = load_and_merge(split="test")
X_test, _ = engineer_model_features(test_df, is_train=False)

# Predict using the best model and optimal threshold
test_proba = predict_proba(best_model, X_test)
test_pred = (test_proba >= optimal_threshold).astype(int)

print(f"Test predictions: {pd.Series(test_pred).value_counts().to_dict()}")

In [ ]:
# Build submission file
submission = pd.DataFrame({
    "customerid": test_df["customerid"].values,
    "Good_Bad_flag": test_pred
})

submission_path = "../outputs/submissions/submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
submission.head()

## 11. Summary of Results

*Fill in after running — key findings on which model performed best, what the optimal threshold was, which features SHAP identified as most important, and any honest limitations of the model.*